# Domain knowledge NLP 

This notebook follows [this](https://huggingface.co/learn/nlp-course/en/chapter7/3) HuggingFace tutorial. 

Using a chunk of data, I fine-tune an existing BERT model to have domain specific knowledge. 

In [53]:
import pandas as pd
import numpy as np
import math
import collections
from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, default_data_collator, TrainingArguments, Trainer, pipeline
import torch
from datasets import Dataset

import sys
sys.path.append('../pipeline')
from comments_saver import CommentsSaver
from nlp_tasks import NLP_Tasks

## Set-up 

In [54]:
model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

In [55]:
nlp = NLP_Tasks(model_checkpoint)

Device set to use mps:0


In [56]:
distilbert_num_parameters = model.num_parameters() / 1000000
print(f"Number of parameters in DistilBERT: {distilbert_num_parameters:.2f}M")

Number of parameters in DistilBERT: 66.99M


This is the text which I'm going to look at the choice of MASK for. 

In [57]:
text = "This is an awful [MASK]."

In [58]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits

# Find the location of [MASK] in the input and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the 5 [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(text.replace("[MASK]", tokenizer.decode([token])))

This is an awful sight.
This is an awful thing.
This is an awful idea.
This is an awful coincidence.
This is an awful mess.


## Load the data for fine-tuning. 

Here I load a chunk of comments on planning applications. 

In [59]:
# # # Load data
# train_df = pd.read_csv('../outputs/train_comments.csv')
# test_df = pd.read_csv('../outputs/test_comments.csv')

# # Convert DataFrame to Hugging Face Dataset
# train_dataset = Dataset.from_pandas(train_df)
# test_dataset = Dataset.from_pandas(test_df)

In [60]:
cs = CommentsSaver(env='prod')
df = cs.read_all()

# Just using the subset of data from Brent (approx 2,000 comments)
df_brent = df[df['council']=='Brent'].reset_index(drop=True)

Connecting to the ai4ci-db database...
Successfully connected to ai4ci-db.


In [61]:
# nlp.split_text_on_newline(df_brent, column='comment_text')

In [62]:
# # Split the text into chunks of 512 tokens

# def split_text(text, max_length=512):
#     # Tokenize the text
#     tokens = tokenizer.tokenize(text)
#     # Split into chunks of max_length
#     chunks = []
#     for i in range(0, len(tokens), max_length):
#         chunk = tokens[i:i + max_length]
#         chunks.append(tokenizer.convert_tokens_to_string(chunk))
#     return chunks 

In [63]:
# split the text into chunks of 512 tokens, with overlap of 20 tokens]
def chunk_with_overlap(text, max_length=512, overlap=20):
    # Tokenize the text
    tokens = tokenizer.tokenize(text)
    # Split into chunks of max_length with overlap
    chunks = []
    for i in range(0, len(tokens), max_length - overlap):
        chunk = tokens[i:i + max_length]
        chunks.append(tokenizer.convert_tokens_to_string(chunk))
    return chunks

In [64]:
def split_text_by_length(df, column='cleaned_text', max_length=512, overlap=20, filter_empty=True, filter_short=True, min_length=5):
    df = df.copy()
    df[column] = df[column].fillna('').astype(str)
    df[column] = df[column].apply(lambda x: chunk_with_overlap(x, max_length, overlap))

    # Explode the DataFrame to have one row per chunk
    df = df.explode(column, ignore_index=True)

    # Strip whitespace from the resulting chunks
    df[column] = df[column].str.strip()

    # If filter_empty is True, drop any rows where the split chunk is empty
    if filter_empty:
        df = df[df[column] != '']

    # If filter_short is True, drop any rows where the split chunk is shorter than min_length
    if filter_short:
        df = df[df[column].str.len() >= min_length]

    # Reset index after exploding
    df.reset_index(drop=True, inplace=True)

    return df

In [65]:
df_brent = split_text_by_length(df_brent, column='comment_text', max_length=512, overlap=20, filter_empty=True, filter_short=True, min_length=5)

Token indices sequence length is longer than the specified maximum sequence length for this model (1350 > 512). Running this sequence through the model will result in indexing errors


In [66]:
print(f'Max length of string in df_brent: {df_brent["comment_text"].str.len().max()}')

Max length of string in df_brent: 3083


In [67]:
# Convert DataFrame to Hugging Face Dataset
dataset_brent = Dataset.from_pandas(df_brent)

In [68]:
dataset_brent['comment_text']

['objection received by email',
 'obj received by email',
 'obj received by email',
 'obj received by email',
 'obj received by email',
 "i firmly oppose the planned change from residential ( use class : c3 ) to hmo ( use class : c4 ). this area suffers of too many hmo ' s that are not maintained or managed to standard. many landlords are not fit for purpose and the result is overflowing bins, continuously changing residents, run down appearances and heavily littered backyards that are being used for gatherings of all form. hmo ' s, that are purely designed to maximise rental income meet opposition in willesden green. we understand the overall planning strategy for a more inclusive and diverse willesden green is being recalibrated and should lead to a reduction of hmo ' s.",
 "this proposed building is ugly, and totally inappropriate in the conservation area. it is far too close to no. 3 dawlish, and to the rear of gardens in dartmouth road. the surrounding homes would suffer a lack of

## Tokenize the data 

This step tokenizes the data used for training. 

In [69]:
# Tokenization function
def tokenize_function(examples):
    result = tokenizer(examples["comment_text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

# Use batched=True to activate fast multithreading!
tokenized_datasets = dataset_brent.map(tokenize_function, batched=True, remove_columns=["address", "stance", "date", "comment_text", "id", "council", "comment_id", "application_id", "add_date"])

Map: 100%|██████████| 2863/2863 [00:00<00:00, 6236.62 examples/s]


In [70]:
print(f'Tokenizer max length: {tokenizer.model_max_length}')

Tokenizer max length: 512


In [71]:
# Get a sense of the number of tokens in each sample
for idx, sample in enumerate(tokenized_datasets[:3]["input_ids"]):
    print(f"Sample {idx} has {len(sample)} tokens.")

Sample 0 has 6 tokens.
Sample 1 has 7 tokens.
Sample 2 has 7 tokens.


Concatenate and chunk the data according to the maximum chunk size. 

In [72]:
# Set maximum chunk size 
chunk_size=512

def group_texts(examples):
    # Concatenate all the comment texts 
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    # Compute length of concatenated texts 
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the last chunk if it's smaller than chunk_size
    total_length = (total_length // chunk_size) * chunk_size
    # Split by chunks of max_len 
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [73]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map: 100%|██████████| 2863/2863 [00:03<00:00, 804.79 examples/s]


In [74]:
tokenizer.decode(lm_datasets[0]["input_ids"])

"[CLS] objection received by email [SEP] [CLS] obj received by email [SEP] [CLS] obj received by email [SEP] [CLS] obj received by email [SEP] [CLS] obj received by email [SEP] [CLS] i firmly oppose the planned change from residential ( use class : c3 ) to hmo ( use class : c4 ). this area suffers of too many hmo ' s that are not maintained or managed to standard. many landlords are not fit for purpose and the result is overflowing bins, continuously changing residents, run down appearances and heavily littered backyards that are being used for gatherings of all form. hmo ' s, that are purely designed to maximise rental income meet opposition in willesden green. we understand the overall planning strategy for a more inclusive and diverse willesden green is being recalibrated and should lead to a reduction of hmo ' s. [SEP] [CLS] this proposed building is ugly, and totally inappropriate in the conservation area. it is far too close to no. 3 dawlish, and to the rear of gardens in dartmou

Mask some of the tokens for training.

In [75]:
# mlm is the fraction of tokens to mask - 15% is popular in the literature. 
mlm_prob = 0.15

# mask the tokens
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=mlm_prob)

Some examples of what this type of masking looks like.

In [76]:
samples = [lm_datasets[i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(tokenizer.decode(chunk))

[CLS] objection received by email [SEP] [CLS] ob [MASK] received by email [SEP] [CLS] [MASK]j received by email [SEP] [CLS] obj received by email [SEP] [CLS] obj received by email [SEP] [CLS] i aliens oppose the planned change from residential ( use class : c3 ) to hmo ( use class : c4 ). this area [MASK] of too many hmo ' s [MASK] are not maintained or managed to standard. many landlords are not [MASK] for purpose and the [MASK] is overflowing bins, continuously changing [MASK], run down appearances and [MASK] littered [MASK] [MASK] that are being used for gatherings [MASK] all form. elevators [MASK] ' s, that are purely designed to maximise rental income meet opposition in willesden green. we understand the overall planning strategy for a more inclusive and diverse willesden green is being [MASK]alibratednh [MASK] lead to a reduction of hmo [MASK] s. [SEP] [CLS] this proposed building isnse [MASK] and totally inappropriate in the conservation area [MASK] it is far too [MASK] to no. 3

Alternative function for masking - this masks whole words, rather than masking tokens (which could be parts of words). 

In [77]:
wwm_probability = 0.2

def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # create a map between words and corresponding token indices 
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)

        
        # randomly mask whole words 
        mask = np.random.binomial(1, wwm_probability, (len(mapping)))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels

    return default_data_collator(features)

Examples of what this type of masking looks like.

In [78]:
samples = [lm_datasets[i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)

for chunk in batch["input_ids"]:
    print(tokenizer.decode(chunk))

[CLS] objection received by email [SEP] [CLS] obj received by email [SEP] [CLS] obj received by email [SEP] [CLS] obj received by email [SEP] [CLS] obj received [MASK] email [SEP] [CLS] i [MASK] oppose the planned change from residential ( use class [MASK] c3 ) [MASK] hmo ( use class : c4 ). this area suffers of [MASK] many hmo ' s that are not maintained or managed to standard [MASK] [MASK] landlords are not [MASK] for [MASK] and the result [MASK] overflowing bins [MASK] continuously changing residents, [MASK] down [MASK] [MASK] heavily littered backyards that are being used for [MASK] of [MASK] [MASK]. [MASK] [MASK] ' s, that [MASK] [MASK] [MASK] to [MASK] [MASK] rental income meet opposition in willesden green. we understand the overall planning strategy for a [MASK] inclusive and diverse willesden green is being [MASK] [MASK] [MASK] [MASK] and should lead [MASK] a reduction of hmo ' s [MASK] [SEP] [CLS] this proposed building is ugly [MASK] and totally inappropriate [MASK] the cons

## Downsample the data 
NOTE: I've accidentally split the data twice - since I already had a train/test split when I loaded the datasets into the nb. 

In [79]:
downsampled_dataset = lm_datasets.train_test_split(test_size=0.2)

In [80]:
len(downsampled_dataset["train"])

1051

## Train the model!!!

In [81]:
batch_size = 8

logging_steps = len(downsampled_dataset["train"]) // batch_size
training_args = TrainingArguments(
    output_dir="../outputs",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    logging_steps=logging_steps,
    fp16=False,
    bf16=True # Note this enables bfloat16 conversion which is supported by the Apple MPD backend
)

In [82]:
trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer 
)

/var/folders/4n/x6w1yfcx01qbymrsfpz4ybq00000gn/T/ipykernel_45692/1257576995.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [83]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 21.57


In [84]:
# run the training loop!
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,2.625200,2.374133,0.000900
2,2.476100,2.285145,0.000900
3,2.416600,2.262948,0.000900


TrainOutput(global_step=396, training_loss=2.5039611392550998, metrics={'train_runtime': 1720.7944, 'train_samples_per_second': 1.832, 'train_steps_per_second': 0.23, 'total_flos': 417965325170688.0, 'train_loss': 2.5039611392550998, 'epoch': 3.0})

## Model performance 

In [85]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 9.61


See how the model performs on some MASKed sentences.

In [86]:
mask_filler = pipeline("fill-mask", model=model, tokenizer=tokenizer)

preds = mask_filler(text)
for pred in preds:
    print(f"{pred['sequence']}")

Device set to use mps:0


this is an awful sight.
this is an awful place.
this is an awful thing.
this is an awful idea.
this is an awful experience.


In [87]:
text1 = "I [MASK] this planning [MASK]"
mask_filler = pipeline("fill-mask", model=model, tokenizer=tokenizer)


while "[MASK]" in text1:
    preds = mask_filler(text1)  # Get predictions for the first [MASK]
    
    if isinstance(preds, list) and isinstance(preds[0], list):  # Check if multiple masks were processed at once
        preds = preds[0]  # Take only the first mask's predictions

    best_pred = preds[0]["token_str"]  # Choose the top predicted word
    text1 = text1.replace("[MASK]", best_pred, 1)  # Replace only the first [MASK]

print(text1)

Device set to use mps:0


I need this planning .


In [88]:
preds

[{'score': 0.5745265483856201,
  'token': 1012,
  'token_str': '.',
  'sequence': 'i need this planning.'},
 {'score': 0.07595298439264297,
  'token': 2832,
  'token_str': 'process',
  'sequence': 'i need this planning process'},
 {'score': 0.06490267813205719,
  'token': 6656,
  'token_str': 'permission',
  'sequence': 'i need this planning permission'},
 {'score': 0.03801484778523445,
  'token': 4646,
  'token_str': 'application',
  'sequence': 'i need this planning application'},
 {'score': 0.037809230387210846,
  'token': 1025,
  'token_str': ';',
  'sequence': 'i need this planning ;'}]

## Save the model 

This saves the models weights and parameters so it can be used for downstream tasks. 

In [89]:
# model.save_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased")
# tokenizer.save_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased")